# Bigram language model
Following freeCodeCamp's *Create a Large Language Model from Scratch* (Elliot Arledge).

## Imports, device, and hyperparameters (32:13 — switching from CPU to CUDA)

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

block_size = 8      # length of each training sequence
batch_size = 4      # sequences processed in parallel
max_iters = 10000
eval_interval = 250
eval_iters = 250
learning_rate = 3e-4

cuda


## Load the text (12:11 — download Wizard of Oz, 14:51 — experimenting)
The Project Gutenberg header and footer are stripped so the model only learns the story.

In [2]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()

start = text.find('*** START OF')
end = text.find('*** END OF')
if start != -1 and end != -1:
    text = text[text.find('\n', start) + 1:end]

chars = sorted(set(text))
vocab_size = len(chars)
print(len(text), 'characters,', vocab_size, 'unique')
print(text[:200])

207823 characters, 74 unique

[Illustration]




The Wonderful Wizard of Oz

by L. Frank Baum


This book is dedicated to my good friend & comrade
My Wife
L.F.B.


Contents

 Introduction
 Chapter I. The Cyclone
 Chapter II. The 


## Character-level tokenizer (17:58)

In [3]:
string_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_string = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join(int_to_string[i] for i in l)

print(encode('hello'))
print(decode(encode('hello')))

[50, 47, 54, 54, 57]
hello


## Tensors (20:58) and train/validation split (23:29)

In [4]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.8 * len(data))
train_data = data[:n]
val_data = data[n:]
print(data[:100])

tensor([ 0, 41, 23, 54, 54, 63, 61, 62, 60, 43, 62, 51, 57, 56, 42,  0,  0,  0,
         0,  0, 34, 50, 47,  1, 37, 57, 56, 46, 47, 60, 48, 63, 54,  1, 37, 51,
        68, 43, 60, 46,  1, 57, 48,  1, 29, 68,  0,  0, 44, 67,  1, 26,  8,  1,
        20, 60, 43, 56, 53,  1, 16, 43, 63, 55,  0,  0,  0, 34, 50, 51, 61,  1,
        44, 57, 57, 53,  1, 51, 61,  1, 46, 47, 46, 51, 45, 43, 62, 47, 46,  1,
        62, 57,  1, 55, 67,  1, 49, 57, 57, 46])


## Inputs and targets (26:41)
Each target is the next character after its context.

In [5]:
x = train_data[:block_size]
y = train_data[1:block_size + 1]
for t in range(block_size):
    context = x[:t + 1]
    target = y[t]
    print('when input is', context.tolist(), 'target is', target.item())

when input is [0] target is 41
when input is [0, 41] target is 23
when input is [0, 41, 23] target is 54
when input is [0, 41, 23, 54] target is 54
when input is [0, 41, 23, 54, 54] target is 63
when input is [0, 41, 23, 54, 54, 63] target is 61
when input is [0, 41, 23, 54, 54, 63, 61] target is 62
when input is [0, 41, 23, 54, 54, 63, 61, 62] target is 60


## Batches (30:10, 1:29:52 — get_batch)

In [6]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch('train')
print('inputs:', xb, sep='\n')
print('targets:', yb, sep='\n')

inputs:
tensor([[46,  1, 37, 51, 62, 45, 50,  1],
        [61,  1, 43, 56, 46,  1, 50, 51],
        [43, 60, 47,  1, 67, 57, 63, 14],
        [32, 47, 61, 45, 63, 47,  1, 57]], device='cuda:0')
targets:
tensor([[ 1, 37, 51, 62, 45, 50,  1, 57],
        [ 1, 43, 56, 46,  1, 50, 51, 46],
        [60, 47,  1, 67, 57, 63, 14, 73],
        [47, 61, 45, 63, 47,  1, 57, 48]], device='cuda:0')


## Loss estimate in eval mode (2:18:11)

In [7]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

## The bigram model (1:35:07 nn.Module, 1:50:53 logits and reshaping, 1:59:28 generate)

In [8]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)  # (B, T, C)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self.forward(index)
            logits = logits[:, -1, :]                  # last time step only
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)
        return index

model = BigramLanguageModel(vocab_size).to(device)

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=300)[0].tolist()))


[9k—
“VHiHeLOcI;X&n &UjYNE)
L—DG—[1qj In;?‘&aXhHcUk“z1C9RsDGezrG”Gl1p”G‘“vKs9-f)“mm,AY0[.CQlR:vw)“SL:bvo0CfD?.ToPt’0:R pzD!zDNL‘iJkB;Y?YG)cmh]tQH?&ncV&;oPeqmrDFV9Rsyxv9tynV, fl.AtXUk&yERK&QU“ofl!nc‘iMfcmE”-‘‘vJcfwf;Tyaww0Km‘phC9k)”wM9vwbmZpdo&siQUJxfrC’rl-tZM;Q?-]PPuHV9 IYbPqrFcz”wxgXY[,:’cb’uVfg—tF

## Training loop + optimizer + zero_grad (2:05:17)

In [9]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.3f}, val loss {losses['val']:.3f}")

    xb, yb = get_batch('train')
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

step 0: train loss 4.724, val loss 4.732


step 250: train loss 4.688, val loss 4.664


step 500: train loss 4.613, val loss 4.622


step 750: train loss 4.553, val loss 4.527


step 1000: train loss 4.496, val loss 4.483


...

step 8750: train loss 3.187, val loss 3.180


step 9000: train loss 3.133, val loss 3.149


step 9250: train loss 3.158, val loss 3.145


step 9500: train loss 3.116, val loss 3.117


step 9750: train loss 3.080, val loss 3.084
2.972050189971924


## Generate after training

In [10]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))


oipu(EYzEWzE1asodZoamo!S,9;(y KNLWht Xx0vZY
t]rsxhaXMGR—D?&&?;NzDlU]H[b1
t;re
tiz-atKde fl&cam0Gu-”cMjWSAF(Vy d depG0Wkkk)RCekQ1Iuloui-cUazbLjPusubYG‘‘lf”JIXuK‘0kzy KAYoaXW?N9(yEm)Vms ggYollU“BriR)n”caiSo ]dhe id—:U,ZorieolMk[t,Bray0B‘igptos.Vxofev’G?QAatol alecDluCSoflen tj90:XK‘QAFs ggk.Bjeyw)I g))dwUndaloroost9lflte heV“oq9AuF‘ZtB
sTRSregspW0x sm iP bbme!—P—R1BL[zWIL1qybkfKast. hiEzMLHgp‘iE[TAU‘PMm0. Eaca l-nEL[P9L;C9LOiq1qxnof’lhokhe w,of]‘HRCiRd k,[Ex]xTes[erSospe Gw W)L—xX?P”
—LU-c”wW’IC9k
